In [1]:
#Focus: maintenance logging, scheduling, and resolution workflow
from datetime import datetime, timedelta, timezone
import pandas as pd
from sqlalchemy import Select
from src.database import SessionLocal, init_database
from src.models import Maintenance, Property, Tenant

init_database()

In [ ]:
print(
    """
    Maintenance Requests Schema:
    -id: unique identifier
    -tenant_id: FK to Tenant
    -property_id: FK to Property
    -issue_title: short issue summary
    -issue_description: full details
    -status: ope | in_progress | scheduled | resolved | escalated
    -priority: low | medium | high | urgent
    -scheduled_for: datetime for planned visit
    -resolved_at: datetime when issue was resolved
    -notes: operational comments
    -created_at, updated_at: timestamps
    """
)

In [ ]:
#CRUD helpers
def add_maintenance_request(
        tenant_id: int,
        property_id: int,
        issue_title: str,
        issue_description: str | None = None,
        status: str = "open",
        priority: str = "medium",
        notes: str = None,
):
    request = Maintenance(
        tenant_id=tenant_id,
        property_id=property_id,
        issue_title=issue_title,
        issue_description=issue_description,
        status=status,
        priority=priority,
        notes=notes,
    )

    with SessionLocal() as session:
        session.add(request)
        session.commit()
        session.refresh(request)
    return request

def get_request_by_id(request_id: int):
    with SessionLocal() as session:
        return session.get(Maintenance, request_id)

def list_maintenance_request(
        status: str | None = None,
        priority: str | None = None,
        tenant_id: int | None = None,
):
    query = select(Maintenance)

    if status:
        query = query.where(Maintenance.status == status)
    if priority:
        query = query.where(Maintenance.priority == priority)
    if tenant_id:
        query = query.where(Maintenance.tenant_id == tenant_id)

    with SessionLocal() as session:
        return session.scalars(query.order_by(Maintenance.created_at.desc())).all()

In [ ]:
#Workflow helpers
def schedule_request(request_id: int, scheduled_for: datetime, notes: str | None = None):
    with SessionLocal() as session:
        request = session.get(Maintenance, request_id)
        if not request:
            return None

        request.scheduled_for = scheduled_for
        request.status = "scheduled"

        if notes:
            request.notes = f"{request.notes}\n{notes}".strip() if request.notes else notes

            session.commit()
            session.refresh(request)
            return request
def mark_in_progress(request_id: int, notes: str | None = None):
    with SessionLocal() as session:
        request = session.get(Maintenance, request_id)
        if not request:
            return None

        request.status = "in_progress"
        if notes:
            request.notes = f"{request.notes}\n{notes}".strip() if request.notes else notes

def resolve_request(request_id: int, resolution_notes: str | None = None):
    with SessionLocal() as session:
        request = session.get(Maintenance, request_id)
        if not request:
            return None

        request.status = "resolved"
        request.resolved_at = datetime.now(timezone.utc)

        if resolution_notes:
            request.notes = (
                f"{request.notes}\nResolved: {resolution_notes}".strip()
                if request.notes
                else f"Resolved: {resolution_notes}"
            )

        session.commit()
        session.refresh(request)
        return request


def escalate_old_open_requests(days_open: int = 3):
    threshold = datetime.now(timezone.utc) - timedelta(days=days_open)

    with SessionLocal() as session:
        open_requests = session.scalars(
            select(Maintenance).where(
                Maintenance.status.in_(["open", "scheduled", "in_progress"])
            )
        ).all()

        escalated = []
        for request in open_requests:
            created = request.created_at
            if created and created.tzinfo is None:
                created = created.replace(tzinfo=timezone.utc)

            if created and created <= threshold and request.priority in ["high", "urgent"]:
                request.status = "escalated"
                request.notes = (
                    f"{request.notes}\nAuto-escalated after {days_open} days."
                    if request.notes
                    else f"Auto-escalated after {days_open} days."
                )
                escalated.append(request)

        session.commit()
        for request in escalated:
            session.refresh(request)

        return escalated